# 01. Phân tích Dữ liệu Châu Á Hợp nhất & Cân bằng (AFAD-Full + FairFace + UTKFace + AAF)
Notebook này chạy **Local** trên bộ dữ liệu Châu Á duy nhất tại `dataset/processed/` (16.522 ảnh, `0=Female / 1=Male`, 9 nhóm tuổi `0-2` … `70+`):
1. Kiểm tra tính toàn vẹn, hợp lệ nhãn và xác nhận **0 rò rỉ dữ liệu** giữa `Train / Val / Test` từ `dataset/metadata/`.
2. Thống kê chi tiết theo **9 nhóm tuổi (`age_group`) × Giới tính**, **4 nguồn dữ liệu**, và **nhóm sắc tộc Châu Á (`Asian`, `East Asian`, `Southeast Asian`)**.
3. Trực quan hóa phân phối dữ liệu và hiển thị ảnh mẫu theo nhóm tuổi, giới tính, nguồn.
4. Kiểm tra kích thước, tỷ lệ khung hình, chất lượng ảnh (mờ/tối/sáng) và Data Augmentation `224×224`.
5. Đóng gói `dataset/processed` thành `dataset_processed.zip` để upload lên `MyDrive/dataset_processed.zip` chạy trên Google Colab.

In [ ]:
import os, sys

cur = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(cur, "src")) and cur != os.path.dirname(cur):
    cur = os.path.dirname(cur)

os.chdir(cur)
if cur not in sys.path:
    sys.path.insert(0, cur)

print("Project Root:", os.getcwd())
print("dataset/processed exists:", os.path.isdir("dataset/processed"))

In [ ]:
# Đọc các file splits & kiểm tra tính hợp lệ + chống rò rỉ dữ liệu
import json
import pandas as pd

train_df = pd.read_csv("dataset/metadata/train.csv")
val_df = pd.read_csv("dataset/metadata/val.csv")
test_df = pd.read_csv("dataset/metadata/test.csv")
full_df = pd.concat(
    [train_df.assign(split="Train"), val_df.assign(split="Val"), test_df.assign(split="Test")],
    ignore_index=True,
)

print(f"Tổng số ảnh sau hợp nhất & cân bằng: {len(full_df):,} ảnh")
print(f" - Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Kiểm tra hợp lệ nhãn & chống rò rỉ
print("Missing values:", full_df[["image_path", "age", "age_group", "gender", "source", "race"]].isna().sum().to_dict())
print("Gender labels:", sorted(full_df["gender"].unique().tolist()), "(0=Female, 1=Male)")
print("Age min -> max:", full_df["age"].min(), "->", full_df["age"].max())

leak_path = len(set(train_df["image_path"]) & set(val_df["image_path"])) + len(set(train_df["image_path"]) & set(test_df["image_path"])) + len(set(val_df["image_path"]) & set(test_df["image_path"]))
leak_md5 = len(set(train_df["md5"]) & set(val_df["md5"])) + len(set(train_df["md5"]) & set(test_df["md5"])) + len(set(val_df["md5"]) & set(test_df["md5"]))
print(f"Kiểm tra rò rỉ giữa Train/Val/Test -> Trùng path: {leak_path} | Trùng MD5: {leak_md5}")

if os.path.exists("dataset/metadata/norm_stats.json"):
    print("Train Norm Stats (224x224):", json.load(open("dataset/metadata/norm_stats.json", encoding="utf-8")))
if os.path.exists("dataset/metadata/age_scaler.json"):
    print("Age Scaler:", json.load(open("dataset/metadata/age_scaler.json", encoding="utf-8")))

In [ ]:
# Thống kê chi tiết: 9 nhóm tuổi x Giới tính, 4 Nguồn dữ liệu, và Chủng tộc
AGE_GROUPS_9 = ["0-2", "3-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]

print("=== 1. PHÂN BỐ 9 NHÓM TUỔI x GIỚI TÍNH (SAU FLEXIBLE CAP = 2000) ===")
ct_age_g = pd.crosstab(full_df["age_group"], full_df["gender_str"], margins=True).reindex(AGE_GROUPS_9 + ["All"])
display(ct_age_g)

print("\n=== 2. PHÂN BỐ THEO 4 NGUỒN DỮ LIỆU x TẬP SPLIT ===")
display(pd.crosstab(full_df["source"], full_df["split"], margins=True)[["Train", "Val", "Test", "All"]])

print("\n=== 3. PHÂN BỐ CHỦNG TỘC (RACE) ===")
race_tbl = full_df["race"].value_counts().to_frame(name="Số ảnh")
race_tbl["Tỷ lệ (%)"] = (race_tbl["Số ảnh"] / len(full_df) * 100).round(2)
display(race_tbl)

In [ ]:
# Trực quan hóa tổng thể dữ liệu hợp nhất (9 nhóm tuổi, nguồn, chủng tộc, phân bố tuổi)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. 9 nhóm tuổi theo giới tính
ct_plot = pd.crosstab(full_df["age_group"], full_df["gender_str"]).reindex(AGE_GROUPS_9)
ct_plot[["Male", "Female"]].plot(kind="bar", ax=axes[0, 0], color=["#1d4ed8", "#db2777"])
axes[0, 0].set_title("Phân bố 9 Nhóm tuổi x Giới tính (Flexible Cap = 2000)")
axes[0, 0].set_xlabel("Nhóm tuổi"); axes[0, 0].set_ylabel("Số ảnh")
axes[0, 0].tick_params(axis="x", rotation=25)

# 2. Đóng góp từ 4 nguồn dữ liệu
src_counts = full_df["source"].value_counts()
bars = axes[0, 1].bar(src_counts.index, src_counts.values, color=["#2563eb", "#059669", "#d97706", "#7c3aed"])
for b in bars:
    axes[0, 1].text(b.get_x() + b.get_width()/2, b.get_height() + 200, f"{int(b.get_height()):,}", ha="center", fontweight="bold")
axes[0, 1].set_title("Số lượng ảnh theo từng Nguồn dữ liệu")
axes[0, 1].set_ylabel("Số ảnh")

# 3. Phân bố Chủng tộc (Race)
race_counts = full_df["race"].value_counts()
axes[1, 0].barh(race_counts.index[::-1], race_counts.values[::-1], color="#0ea5e9")
axes[1, 0].set_title("Phân bố theo Chủng tộc (Race)")
axes[1, 0].set_xlabel("Số ảnh")

# 4. Phân bố tuổi số Nam vs Nữ
axes[1, 1].hist(full_df[full_df["gender"] == 0]["age"], bins=30, alpha=0.55, label="Female (0)", color="#db2777")
axes[1, 1].hist(full_df[full_df["gender"] == 1]["age"], bins=30, alpha=0.55, label="Male (1)", color="#1d4ed8")
axes[1, 1].set_title("Phân bố Tuổi số (1 -> 116) theo Giới tính")
axes[1, 1].set_xlabel("Tuổi"); axes[1, 1].set_ylabel("Số ảnh"); axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Hiển thị ảnh mẫu đại diện cho 9 nhóm tuổi x 2 giới tính (18 ảnh)
from PIL import Image

fig, axes = plt.subplots(2, 9, figsize=(19, 4.8))
for r_idx, g_str in enumerate(["Male", "Female"]):
    for c_idx, ag in enumerate(AGE_GROUPS_9):
        sub = full_df[(full_df["age_group"] == ag) & (full_df["gender_str"] == g_str)]
        row = sub.sample(1, random_state=42).iloc[0]
        with Image.open(row["image_path"]) as im:
            axes[r_idx, c_idx].imshow(im.convert("RGB"))
        axes[r_idx, c_idx].set_title(f"{ag}\n{g_str} | {row['source']}", fontsize=8)
        axes[r_idx, c_idx].axis("off")

plt.suptitle("Ảnh mẫu đại diện theo 9 Nhóm tuổi x Giới tính (Hàng trên: Male | Hàng dưới: Female)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Hiển thị ảnh mẫu theo 4 nguồn dữ liệu (AFAD-Full, FairFace, UTKFace, AAF)
sources = ["AFAD-Full", "FairFace", "UTKFace", "AAF"]
fig, axes = plt.subplots(4, 6, figsize=(14, 9.5))
for r_idx, src in enumerate(sources):
    sub = full_df[full_df["source"] == src].sample(6, random_state=42)
    for c_idx, row in enumerate(sub.itertuples()):
        with Image.open(row.image_path) as im:
            axes[r_idx, c_idx].imshow(im.convert("RGB"))
        axes[r_idx, c_idx].set_title(f"{src} | {row.age_group}\n{row.gender_str} | {row.race}", fontsize=8)
        axes[r_idx, c_idx].axis("off")

plt.suptitle("Ảnh mẫu theo từng nguồn dữ liệu trong tập hợp nhất", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra kích thước ảnh, tỷ lệ khung hình & chất lượng ảnh trên dữ liệu hợp nhất
import cv2
import numpy as np

QC_SAMPLE = 2000
qc_df = full_df.sample(min(QC_SAMPLE, len(full_df)), random_state=42).copy().reset_index(drop=True)

widths, heights, blur_scores, brightness_vals = [], [], [], []
for p in qc_df["image_path"]:
    img_bgr = cv2.imread(p)
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    widths.append(w); heights.append(h)
    blur_scores.append(cv2.Laplacian(gray, cv2.CV_64F).var())
    brightness_vals.append(gray.mean())

qc_df["width"] = widths
qc_df["height"] = heights
qc_df["aspect_ratio"] = np.round(np.array(widths) / np.array(heights), 3)
qc_df["blur_var"] = blur_scores
qc_df["brightness"] = brightness_vals

print(f"=== THỐNG KÊ KÍCH THƯỚC & CHẤT LƯỢNG (mẫu {len(qc_df):,} ảnh hợp nhất) ===")
display(qc_df[["width", "height", "aspect_ratio", "blur_var", "brightness"]].describe().round(2))
print(f"Ảnh mờ (blur_var < 50): {(qc_df['blur_var'] < 50).sum()} ({(qc_df['blur_var'] < 50).mean()*100:.2f}%)")
print(f"Ảnh tối (brightness < 50): {(qc_df['brightness'] < 50).sum()} ({(qc_df['brightness'] < 50).mean()*100:.2f}%)")
print(f"Ảnh sáng chói (brightness > 210): {(qc_df['brightness'] > 210).sum()} ({(qc_df['brightness'] > 210).mean()*100:.2f}%)")

In [ ]:
# Kiểm tra Data Augmentation 224x224 + đọc Mean/Std từ norm_stats.json
import numpy as np
import torch
from torchvision import transforms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from src.datasets.afad_dataset import AFADDataset
from src.datasets.transforms import get_transforms, load_norm_stats

os.makedirs("results/eda", exist_ok=True)
mean, std = load_norm_stats()
print("Đang dùng Mean/Std:", mean, std)

# 1. Biểu diễn t-SNE theo 9 nhóm tuổi & giới tính (~1.200 ảnh mẫu cân bằng)
tsne_parts = [grp.sample(min(70, len(grp)), random_state=42) for _, grp in full_df.groupby(["age_group", "gender"])]
tsne_df = pd.concat(tsne_parts, ignore_index=True)
X = np.vstack([np.asarray(Image.open(p).convert("RGB").resize((32, 32)), dtype=np.float32).flatten() / 255.0 for p in tsne_df["image_path"]])
X_2d = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(PCA(n_components=50, random_state=42).fit_transform(X))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cmap_age = plt.colormaps["tab10"]
for idx, ag in enumerate(AGE_GROUPS_9):
    m = tsne_df["age_group"] == ag
    axes[0].scatter(X_2d[m, 0], X_2d[m, 1], s=14, alpha=0.7, label=ag, color=cmap_age(idx))
axes[0].set_title("t-SNE theo 9 Nhóm tuổi"); axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
for g_lbl, name, col in [(0, "Female (0)", "#db2777"), (1, "Male (1)", "#1d4ed8")]:
    m = tsne_df["gender"] == g_lbl
    axes[1].scatter(X_2d[m, 0], X_2d[m, 1], s=14, alpha=0.65, label=name, color=col)
axes[1].set_title("t-SNE theo Giới tính"); axes[1].legend()
plt.tight_layout(); plt.savefig("results/eda/tsne_age_gender.png", dpi=150, bbox_inches="tight"); plt.show()

# 2. Hình ảnh sau Resize (224x224) & Geometric (Flip + Rotate)
samples = full_df.sample(5, random_state=7)
resize_tf = transforms.Resize((224, 224))
geom_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1.0), transforms.RandomRotation(15)])
fig, axes = plt.subplots(3, 5, figsize=(14, 8.5))
for c, row in enumerate(samples.itertuples()):
    with Image.open(row.image_path) as im:
        im_rgb = im.convert("RGB")
        for r, (img_s, tag) in enumerate([(im_rgb, f"{im_rgb.size[0]}x{im_rgb.size[1]}"), (resize_tf(im_rgb), "224x224"), (geom_tf(im_rgb), "Flip+Rot")]):
            axes[r, c].imshow(img_s); axes[r, c].set_title(f"{row.source} | {row.age_group} ({tag})", fontsize=8); axes[r, c].axis("off")
plt.suptitle("Ảnh gốc -> Resize (224x224) -> Geometric (Flip + Rotate)", fontsize=12)
plt.tight_layout(); plt.savefig("results/eda/preprocess_resize_geometric.png", dpi=150); plt.show()

# 3. Hình ảnh sau Chuẩn hóa (Normalize) & Data Augmentation (224x224)
train_tf, val_tf = get_transforms(img_size=224, augment=True, vflip=False)
loader = DataLoader(AFADDataset("dataset/metadata/train.csv", transform=train_tf), batch_size=8, shuffle=True)
imgs, genders, ages = next(iter(loader))
print("Batch shape:", imgs.shape, "| dtype:", imgs.dtype, "| min/max:", round(imgs.min().item(), 2), round(imgs.max().item(), 2))

mean_arr, std_arr = np.array(mean), np.array(std)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(5):
    t_img = imgs[i].permute(1, 2, 0).numpy()
    axes[0, i].imshow(np.clip(t_img, 0, 1)); axes[0, i].set_title(f"Normalized Tensor", fontsize=8); axes[0, i].axis("off")
    axes[1, i].imshow(np.clip(t_img * std_arr + mean_arr, 0, 1)); axes[1, i].set_title(f"{'Female' if genders[i].item()==0 else 'Male'} | Age: {ages[i].item():.1f}", fontsize=8); axes[1, i].axis("off")
plt.suptitle("Ảnh sau Chuẩn hóa (Normalized Tensor) & Augmentation (224x224)", fontsize=12)
plt.tight_layout(); plt.savefig("results/eda/preprocess_norm_augment.png", dpi=150); plt.show()

In [ ]:
# Đóng gói toàn bộ dự án (code + metadata + 16.522 ảnh processed, bỏ raw/) thành 1 file DUY NHẤT: FINAL-PROJECT.zip
import zipfile

ZIP_PATH = "FINAL-PROJECT.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("."):
        dirs[:] = [d for d in dirs if d not in {".git", "__pycache__", ".ipynb_checkpoints"}]
        rel_root = os.path.relpath(root, ".")
        if rel_root.startswith(os.path.join("dataset", "raw")):
            continue
        for fn in files:
            if fn.endswith(".zip") or fn.endswith(".pyc"):
                continue
            fp = os.path.join(root, fn)
            arc = os.path.normpath(os.path.join("FINAL-PROJECT", rel_root, fn))
            zf.write(fp, arc)

size_mb = os.path.getsize(ZIP_PATH) / (1024 ** 2)
print(f"Đã tạo 1 file zip duy nhất: {os.path.abspath(ZIP_PATH)} ({size_mb:.1f} MB)")
print("Chỉ cần upload đúng file này lên MyDrive/FINAL-PROJECT.zip và mở 02_train_eval_resnet18.ipynb trên Colab!")